## 1. Instalación de dependencias

In [1]:
# pip install curl_cffi beautifulsoup4 pandas

## 2. Imports y funciones auxiliares

In [2]:
import re
import os
import time
import pandas as pd
from bs4 import BeautifulSoup
from curl_cffi import requests as cf_requests

# ──────────────────────────────────────────────
# UTILIDADES
# ──────────────────────────────────────────────

def clean_text(text):
    if not text: return None
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def parse_price(price_text):
    price_text = clean_text(price_text)
    precio = "Consultar"
    expensas = None
    p_match = re.search(r'(USD|ARS|\$)\s?[\$]?\s?([\d\.]+)', price_text)
    if p_match:
        precio = p_match.group(0).strip()
    e_match = re.search(r'[Ee]xp[^$\d]*([\d\.]+)', price_text)
    if not e_match:
        e_match = re.search(r'\+\s?\$?\s?([\d\.]+)', price_text)
    if e_match:
        expensas = e_match.group(0).strip()
    return precio, expensas

def extract_smart_features(row):
    texto = (str(row.get('Descripción', '')) + " " + str(row.get('Detalles', ''))).lower()
    return pd.Series({
        "Amenities":         1 if any(x in texto for x in ["amenities", "piscina", "pileta", "sum", "parrilla", "gym", "sauna", "laundry"]) else 0,
        "Losa_Central":      1 if any(x in texto for x in ["losa radiante", "calefacción central", "caldera central", "piso radiante"]) else 0,
        "Aire_Acond":        1 if any(x in texto for x in ["aire acondicionado", "split", " a/c", "frío-calor"]) else 0,
        "Apto_Credito":      1 if "apto crédito" in texto or "apto credito" in texto else 0,
        "Cochera":           1 if any(x in texto for x in ["cochera", "espacio guarda coche", "estacionamiento", "guarda coche"]) else 0,
        "Seguridad":         1 if any(x in texto for x in ["vigilancia", "seguridad 24", "tótem", "totem", "encargado"]) else 0,
        "Luminoso":          1 if any(x in texto for x in ["luminoso", "todo luz", "vista abierta", "vista panorámica", "sol"]) else 0,
        "Balcon_Aterrazado": 1 if "aterrazado" in texto or "balcón terraza" in texto else 0,
    })


MESES = r'(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|octubre|noviembre|diciembre)'

def parse_address(address_raw):
    """
    Maneja todos los casos conocidos de ZonaProp:
    - 'Bolivia al 4400'                          → calle=Bolivia, altura=4400
    - 'Av Luis María Campos 1500'                → calle=Av Luis María Campos, altura=1500
    - 'Junín 1615 piso 13'                       → calle=Junín, altura=1615, piso=13
    - 'SAN JOSE 445. Entre Belgrano y Venezuela' → calle=SAN JOSE, altura=445
    - '11 de Septiembre de 1888 2231'            → calle=11 de Septiembre de 1888, altura=2231
    - 'Torres del Yacht - Juana Manso al 600'    → calle=Juana Manso, altura=600
    - 'Alvear Tower - Azucena Villaflor'         → calle=None, altura=None (sin número)
    - 'El Faro - 3 Ambientes + Dependencia'      → calle=None, altura=None (número no es altura)
    """
    calle = altura = piso = None
    if not address_raw:
        return calle, altura, piso

    try:
        # 1. Limpiar "al " antes de números
        cleaned = re.sub(r'\bal\s+', '', address_raw, flags=re.IGNORECASE)

        # 2. Si tiene guión, quedarse con el fragmento que tenga número de altura válido
        #    Ej: "Alvear Tower - Azucena Villaflor" → ningún fragmento tiene número → None
        #    Ej: "Torres del Yacht - Juana Manso 600 - 2 Ambientes" → "Juana Manso 600"
        #    Ej: "El Faro - 3 Ambientes" → el "3" no es altura válida (< 10), descartar
        if '-' in cleaned:
            fragmentos = [f.strip() for f in cleaned.split('-')]
            candidato = None
            for frag in fragmentos:
                # Buscar fragmento con número >= 100 (alturas reales de CABA son >= 100)
                if re.search(r'\b\d{3,5}\b', frag):
                    candidato = frag
                    break
            if candidato:
                cleaned = candidato
            else:
                # Ningún fragmento tiene altura válida → nombre de edificio sin dirección
                return None, None, None

        # 3. Intentar capturar piso: "Junín 1615 piso 13"
        match_piso = re.search(
            r'([A-Za-záéíóúÁÉÍÓÚñÑü\s\.]+?)\s+(\d{2,5})\s+piso\s+(\d+)',
            cleaned, re.IGNORECASE
        )
        if match_piso:
            num = int(match_piso.group(2))
            if not (1800 <= num <= 1950 and re.search(MESES, cleaned[:match_piso.start(2)], re.IGNORECASE)):
                calle  = match_piso.group(1).strip().strip('-').strip('.')
                altura = match_piso.group(2).strip()
                piso   = match_piso.group(3).strip()
                return calle, altura, piso

        # 4. Iterar todos los números y encontrar la altura real
        for m in re.finditer(r'(\d{2,5})(?:[.,\s\-]|$)', cleaned):
            num = int(m.group(1))

            # Descartar números < 100: probablemente ambientes, m², baños, etc.
            if num < 100:
                continue

            # Descartar años históricos si hay un mes en el contexto previo
            contexto_previo = cleaned[:m.start()].lower()
            es_año_historico = (1800 <= num <= 1950) and bool(re.search(MESES, contexto_previo, re.IGNORECASE))
            if es_año_historico:
                continue

            # Este número es la altura real
            calle_raw = cleaned[:m.start()].strip().strip('-').strip('.').strip()
            if not calle_raw:
                continue

            calle  = calle_raw
            altura = str(num)
            # piso queda None
            return calle, altura, piso

    except:
        pass

    return None, None, None

def parse_card(item):
    # Link
    href = item.get('data-to-posting', '')
    if not href:
        a = item.find('a', href=re.compile(r'/propiedades/'))
        href = a['href'] if a else ''
    link = f"https://www.zonaprop.com.ar{href}" if href.startswith('/') else href

    # Precio
    price_tag = item.find(attrs={"data-qa": "POSTING_CARD_PRICE"})
    precio_raw = clean_text(price_tag.text) if price_tag else ""
    precio, _ = parse_price(precio_raw)

    # Expensas
    exp_tag = item.find(attrs={"data-qa": "expensas"})
    expensas = clean_text(exp_tag.text) if exp_tag else None

    # ── Dirección: calle+altura y barrio van en tags separados ──
    addr_tag = item.find(class_=re.compile(r'location-address'))
    raw_address = clean_text(addr_tag.text) if addr_tag else ""
    calle, altura, piso = parse_address(raw_address)

    # Barrio/zona (el data-qa="POSTING_CARD_LOCATION")
    barrio_tag = item.find(attrs={"data-qa": "POSTING_CARD_LOCATION"})
    barrio = clean_text(barrio_tag.text) if barrio_tag else None

    # Características
    feat_tag = item.find(attrs={"data-qa": "POSTING_CARD_FEATURES"})
    features = clean_text(feat_tag.get_text(separator=' ')) if feat_tag else None

    # Descripción
    desc_tag = item.find(attrs={"data-qa": "POSTING_CARD_DESCRIPTION"})
    desc = clean_text(desc_tag.text) if desc_tag else None

    return link, precio, expensas, calle, altura, piso, barrio, features, desc

print("✅ Funciones utilitarias cargadas.")

✅ Funciones utilitarias cargadas.


## 3. Función principal del scrapper

In [3]:
def run_scrapper_zonaprop(enlace, operacion, max_pages=3):
    BASE = enlace
    OUTPUT_DIR = "output"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    all_data = []
    seen_links = set()

    for page_num in range(1, max_pages + 1):
        url = f"{BASE}.html" if page_num == 1 else f"{BASE}-pagina-{page_num}.html"
        print(f"\n--- PÁGINA {page_num} ---")
        print(f"URL: {url}")

        try:
            r = cf_requests.get(
                url,
                impersonate="chrome120",
                timeout=20,
                headers={
                    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
                    "Accept-Language": "es-AR,es;q=0.9",
                    "Referer": "https://www.zonaprop.com.ar/",
                }
            )
            if r.status_code != 200:
                print(f"❌ Status {r.status_code}. Deteniendo.")
                break
        except Exception as e:
            print(f"❌ Error de red: {e}")
            break

        soup = BeautifulSoup(r.text, 'html.parser')
        items = soup.find_all('div', attrs={"data-qa": "posting PROPERTY"})

        if not items:
            print("⚠️ No se encontraron cards. Fin del listado.")
            break

        print(f"  → {len(items)} propiedades encontradas")

        for item in items:
            try:
                link, precio, expensas, calle, altura, piso, barrio, features, desc = parse_card(item)

                if not link or link in seen_links:
                    continue
                seen_links.add(link)

                print(f"  → {calle} {altura} | {precio}")
                
                all_data.append({
                    "Precio":      precio,
                    "Expensas":    expensas,
                    "Calle":       calle,
                    "Altura":      altura,
                    "Piso":        piso,
                    "Barrio":      barrio,       
                    "Detalles":    features,
                    "Descripción": desc,
                    "Link":        link,
                })

            except Exception as e:
                print(f"    ⚠️ Error en card: {e}")
                continue

        time.sleep(2)

    if not all_data:
        print("⚠️ No se obtuvieron datos.")
        return None

    df = pd.DataFrame(all_data)
    features_df = df.apply(extract_smart_features, axis=1)
    df = pd.concat([df, features_df], axis=1)

    filename = f"zonaprop_{operacion}_{int(time.time())}.tsv"
    filepath = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(filepath, sep='\t', index=False, encoding='utf-8-sig')
    print(f"\n✅ ¡Éxito! Guardado en: {filepath}")
    return df

print("✅ Listo.")

✅ Listo.


## 4. ▶️ Ejecutar el scrapper

# Alquiler

In [4]:
df_alquiler = run_scrapper_zonaprop('https://www.zonaprop.com.ar/departamentos-alquiler-capital-federal', 'alquiler', max_pages=3)


--- PÁGINA 1 ---
URL: https://www.zonaprop.com.ar/departamentos-alquiler-capital-federal.html
  → 30 propiedades encontradas
  → French 2700 | USD 3.300
  → Santos Dumont 3400 | $ 1.200.000
  → Avenida Santa Fe 5100 | USD 450
  → None None | USD 800
  → LA Pampa 5000 | $ 750.000
  → Valentín Gómez 3400 | $ 600.000
  → Víctor Martínez 300 | $ 2.000.000
  → Pedro Goyena 1000 | $ 700.000
  → Azopardo 770 | USD 850
  → Lemos 300 | USD 700
  → Av. Directorio 400 | $ 650.000
  → Uriarte 2200 | USD 950
  → Dr. Emilio Ravignani 1400 | USD 1.100
  → Guatemala 6000 | $ 875.000
  → Av. Rivadavia 6100 | $ 900.000
  → Lola Mora 420 | USD 1.100
  → Scalabrini Ortiz 300 | $ 1.400.000
  → Ruggieri 2800 | USD 1.100
  → Libertador 7700 | USD 890
  → Av. Olazabal 4300 | $ 950.000
  → O´Higgins 1500 | $ 690.000
  → Wenceslao Villafañe 1400 | $ 850.000
  → Rawson 300 | $ 900.000
  → La Pampa 600 | USD 4.000
  → Sanchez de Bustamante Almagro 100 | $ 1.300.000
  → Avenida Riestra 5500 | $ 450.000
  → Charca

In [5]:
if df_alquiler is not None:
    print(f"Total propiedades: {len(df_alquiler)}")
    display(df_alquiler.head(10))

    cols = ["Amenities","Losa_Central","Aire_Acond","Apto_Credito",
            "Cochera","Seguridad","Luminoso","Balcon_Aterrazado"]
    print("\n📊 % con cada característica:")
    print((df_alquiler[cols].mean() * 100).round(1).astype(str) + "%")

Total propiedades: 90


,Precio,Expensas,Calle,Altura,Piso,Barrio,Detalles,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,USD 3.300,$ 203.825 Expensas,French,2700,None,"Recoleta, Capital Federal",87 m² tot. 3 amb. 2 dorm. 2 baños,"Edificio antiguo de estilo francés en esquina,...",https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,1,0
1,$ 1.200.000,$ 306.000 Expensas,Santos Dumont,3400,None,"Chacarita, Capital Federal",63 m² tot. 2 amb. 1 dorm. 1 baño,"Alquiler Dos ambientes, 3er Piso al Frente con...",https://www.zonaprop.com.ar/propiedades/clasif...,1,0,1,0,0,1,1,0
2,USD 450,$ 180.000 Expensas,Avenida Santa Fe,5100,None,"Palermo Hollywood, Palermo",30 m² tot. 1 amb. 1 baño,Impecable monoambiente! Balcón al pulmón de ma...,https://www.zonaprop.com.ar/propiedades/clasif...,1,1,1,0,0,1,1,0
3,USD 800,$ 150.000 Expensas,None,None,None,"Puerto Madero, Capital Federal",35 m² tot. 1 amb. 1 dorm. 1 baño,"Excelente Studio en alquiler sin muebles, en Q...",https://www.zonaprop.com.ar/propiedades/clasif...,1,0,0,0,0,1,1,0
4,$ 750.000,$ 196.000 Expensas,LA Pampa,5000,None,"Villa Urquiza, Capital Federal",55 m² tot. 2 amb. 1 dorm. 1 baño,Alquiler departamento 2 ambientes de 55m2 tota...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,0,0,0,0,1,0
5,$ 600.000,$ 227.000 Expensas,Valentín Gómez,3400,None,"Almagro, Capital Federal",35 m² tot. 2 amb. 1 dorm. 1 baño,Impecable unidad de 2 ambientes totalmente rec...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,1,0,0,1,0,0
6,$ 2.000.000,$ 415.000 Expensas,Víctor Martínez,300,None,"Caballito, Capital Federal",111 m² tot. 4 amb. 4 dorm. 2 baños,Departamento piso en Alquiler: - Piso de 5 amb...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,1,0
7,$ 700.000,$ 76.000 Expensas,Pedro Goyena,1000,None,"Caballito, Capital Federal",42 m² tot. 2 amb. 1 dorm. 1 baño,Lindísimo departamento de 2 ambientes en Barri...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,1,0
8,USD 850,$ 322.000 Expensas,Azopardo,770,None,"Puerto Madero, Capital Federal",37 m² tot. 1 amb. 1 baño,Alquiler temporarioquartier Madero UrbanoMonoa...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,1,0,0,1,1,0
9,USD 700,$ 223.000 Expensas,Lemos,300,None,"Villa Crespo, Capital Federal",50 m² tot. 2 amb. 1 dorm. 2 baños,Corredor Inmobiliario responsable: Arq. Jorge ...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,0,0,0,0,1,0



📊 % con cada característica:
Amenities            55.6%
Losa_Central         11.1%
Aire_Acond           40.0%
Apto_Credito          0.0%
Cochera              30.0%
Seguridad            28.9%
Luminoso             73.3%
Balcon_Aterrazado     5.6%
dtype: object


# Alquiler Temporal

In [6]:
df_alquiler_temporal = run_scrapper_zonaprop('https://www.zonaprop.com.ar/departamentos-alquiler-temporal-capital-federal', 'alquiler_temporal', max_pages=3)


--- PÁGINA 1 ---
URL: https://www.zonaprop.com.ar/departamentos-alquiler-temporal-capital-federal.html


  → 27 propiedades encontradas
  → Av Coronel Diaz 1400 | USD 1.450
  → Avenida Olleros 1700 | $ 500.000
  → Costa Rica 3900 | USD 1.000
  → Roosevelt 2800 | USD 1.500
  → Perón 4100 | USD 600
  → Lafinur 3200 | USD 1.900
  → Arevalo 1400 | USD 800
  → Av. Pueyrredon 2300 | USD 2.200
  → Vera 800 | USD 700
  → Ortega y Gasset 1800 | USD 1.600
  → Arenales 3800 | USD 1.500
  → Guemes 3600 | USD 1.500
  → Ruggieri 2900 | USD 2.200
  → Pacheco de Melo 2700 | $ 650.000
  → Sanchez de Bustamante 2000 | USD 1.200
  → Santa Fe 5381 | USD 1.100
  → Romulo Naon 3500 | USD 600
  → Av. Corrientes 1600 | USD 850
  → BORGES JORGE LUIS 2287 | USD 600
  → MONROE 2671 | USD 750
  → None None | USD 850
  → ECUADOR 1578 | USD 1.100
  → Ayacucho 2100 | USD 1.500
  → SAN JOSE 445 | USD 550
  → MANSILLA 2600 | USD 900
  → Dorrego 2700 | USD 1.050
  → Santa Fe 2200 | USD 1.800

--- PÁGINA 2 ---
URL: https://www.zonaprop.com.ar/departamentos-alquiler-temporal-capital-federal-pagina-2.html
  → 30 propiedades 

In [7]:
if df_alquiler_temporal is not None:
    print(f"Total propiedades: {len(df_alquiler_temporal)}")
    display(df_alquiler_temporal.head(10))

    cols = ["Amenities","Losa_Central","Aire_Acond","Apto_Credito",
            "Cochera","Seguridad","Luminoso","Balcon_Aterrazado"]
    print("\n📊 % con cada característica:")
    print((df_alquiler_temporal[cols].mean() * 100).round(1).astype(str) + "%")

Total propiedades: 87


,Precio,Expensas,Calle,Altura,Piso,Barrio,Detalles,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,USD 1.450,None,Av Coronel Diaz,1400,None,"Palermo, Capital Federal",79 m² tot. 3 amb. 2 dorm. 1 baño,Departamento de tres ambientes en palermo para...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,1,0,0,0,0,0
1,$ 500.000,$ 200.000 Expensas,Avenida Olleros,1700,None,"Las Cañitas, Palermo",25 m² tot. 1 amb. 1 baño,Corredor Responsable: uno bienes raices S. R. ...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,1,0,0,0,0,0
2,USD 1.000,None,Costa Rica,3900,None,"Palermo, Capital Federal",45 m² tot. 1 amb. 1 baño,Departamento de un ambiente para alquiler temp...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,1,0,0,0,1,0
3,USD 1.500,$ 310.000 Expensas,Roosevelt,2800,None,"Belgrano, Capital Federal",70 m² tot. 3 amb. 2 dorm. 1 baño 1 coch.,Alquiler de duplex totalmente amoblado de 3 am...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,1,0,1,0,1,0
4,USD 600,$ 128.000 Expensas,Perón,4100,None,"Almagro, Capital Federal",33 m² tot. 2 amb. 1 dorm. 1 baño,Alquiler temporario de Departamento 2 ambiente...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,1,0,0,0,1,0
5,USD 1.900,None,Lafinur,3200,None,"Palermo Chico, Palermo",99 m² tot. 4 amb. 3 dorm. 1 baño,Alquiler temporarioperiodopreciomayousd1. 900J...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,1,0,0,0,1,0
6,USD 800,$ 110.000 Expensas,Arevalo,1400,None,"Palermo Hollywood, Palermo",31 m² tot. 1 amb. 1 baño,Alquiler Temporario – Loft Amoblado en Palermo...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,1,0,0,1,1,0
7,USD 2.200,None,Av. Pueyrredon,2300,None,"Recoleta, Capital Federal",150 m² tot. 4 amb. 3 dorm. 2 baños,Excelente departamento de cuatro ambientes en ...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,1,0,0,0,0,0
8,USD 700,None,Vera,800,None,"Villa Crespo, Capital Federal",35 m² tot. 1 amb. 1 baño,Alquiler temporario studio en villa crespo. Ub...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,0,0,0,1,1,0
9,USD 1.600,None,Ortega y Gasset,1800,None,"Las Cañitas, Palermo",82 m² tot. 3 amb. 2 dorm. 2 baños 1 coch.,Alquiler temporal en Las Cañitas: Amplio y lum...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,0,0,1,1,1,0



📊 % con cada característica:
Amenities            67.8%
Losa_Central          6.9%
Aire_Acond           64.4%
Apto_Credito          0.0%
Cochera              23.0%
Seguridad            21.8%
Luminoso             59.8%
Balcon_Aterrazado     8.0%
dtype: object


# Venta

In [8]:
df_venta = run_scrapper_zonaprop('https://www.zonaprop.com.ar/departamentos-venta-capital-federal', 'venta', max_pages=3)


--- PÁGINA 1 ---
URL: https://www.zonaprop.com.ar/departamentos-venta-capital-federal.html
  → 25 propiedades encontradas
  → Bogotá 800 | USD 139.500
  → Ciudad de la Paz 3299 | USD 189.900
  → Libertador 3100 | USD 790.000
  → Olazabal 5219 | USD 113.000
  → Quintana 100 | USD 415.000
  → VENEZUELA 2300 | USD 89.500
  → Beruti 3000 | USD 295.000
  → Grecia 3500 | USD 430.200
  → None None | USD 165.000
  → Doblas 900 | USD 130.000
  → Lascano 5248 | USD 73.000
  → O'HIGGINS 2100 | USD 290.000
  → Andrés Lamas 1900 | USD 151.000
  → Rivadavia Av 5800 | USD 150.000
  → Corrientes 3800 | USD 128.500
  → Libertador 7400 | USD 749.000
  → Uriarte 2200 | USD 163.000
  → Virrey del Pino 2800 | USD 260.000
  → Migueletes 570 | USD 218.000
  → Aranguren 2000 | USD 128.000
  → Artigas 400 | USD 360.000
  → Libertador 7400 | USD 1.380.000
  → Anchorena Tomas Manuel de 1700 | USD 115.000
  → Olleros 4144 | USD 88.200
  → None None | USD 120.268

--- PÁGINA 2 ---
URL: https://www.zonaprop.com.ar

In [9]:
if df_venta is not None:
    print(f"Total propiedades: {len(df_venta)}")
    display(df_venta.head(10))

    cols = ["Amenities","Losa_Central","Aire_Acond","Apto_Credito",
            "Cochera","Seguridad","Luminoso","Balcon_Aterrazado"]
    print("\n📊 % con cada característica:")
    print((df_venta[cols].mean() * 100).round(1).astype(str) + "%")

Total propiedades: 75


,Precio,Expensas,Calle,Altura,Piso,Barrio,Detalles,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,USD 139.500,None,Bogotá,800,None,"Caballito, Capital Federal",52 m² tot. 2 amb. 1 dorm. 1 baño,Venta de departamento 2 ambientes en caballito...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,1,0,1,0,1,0
1,USD 189.900,None,Ciudad de la Paz,3299,None,"Núñez, Capital Federal",73 m² tot. 3 amb. 2 dorm. 2 baños,Entrega en noviembre 2027. Excelente oportunid...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,0,0,0,0,1,0
2,USD 790.000,$ 650.000 Expensas,Libertador,3100,None,"Palermo Chico, Palermo",459 m² tot. 5 amb. 3 dorm. 2 baños 1 coch.,Con super jardin propio de 300 M2 -en excelent...,https://www.zonaprop.com.ar/propiedades/clasif...,0,1,0,0,1,1,0,0
3,USD 113.000,$ 106.000 Expensas,Olazabal,5219,None,"Villa Urquiza, Capital Federal",54 m² tot. 3 amb. 2 dorm. 1 baño,Departamento de 3 ambientes en 4° piso al cont...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,1,0
4,USD 415.000,$ 657.000 Expensas,Quintana,100,None,"Recoleta, Capital Federal",172 m² tot. 6 amb. 4 dorm. 3 baños,Quintana entre montevideo Y parera. Semipiso a...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,1,0,0
5,USD 89.500,$ 132.000 Expensas,VENEZUELA,2300,None,"Balvanera, Capital Federal",52 m² tot. 3 amb. 2 dorm. 1 baño,Sup total: 52. 12 m2. Departamento de 3 ambien...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,1,1,0
6,USD 295.000,$ 433.133 Expensas,Beruti,3000,None,"Barrio Norte, Capital Federal",59 m² tot. 2 amb. 1 dorm. 2 baños 1 coch.,Residencia 2 ambientes con amenities de catego...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,1,0,1,1,1,0
7,USD 430.200,None,Grecia,3500,None,"Núñez, Capital Federal",128 m² tot. 4 amb. 3 dorm. 2 baños,Excelente edificio premium ubicado en una de l...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,1,0,1,0,1,0
8,USD 165.000,$ 200.000 Expensas,None,None,None,"Caballito, Capital Federal",85 m² tot. 3 amb. 1 baño,Departamento 3 Ambientes en Planta Baja con Pa...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,1,0,0,0,0
9,USD 130.000,$ 200.000 Expensas,Doblas,900,None,"Caballito, Capital Federal",37 m² tot. 1 amb. 1 baño,Venta departamento monoambiente divisible con ...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,0,0,1,1,1,0



📊 % con cada característica:
Amenities            54.7%
Losa_Central          9.3%
Aire_Acond           32.0%
Apto_Credito         17.3%
Cochera              45.3%
Seguridad            37.3%
Luminoso             73.3%
Balcon_Aterrazado     9.3%
dtype: object
